## **Intro to Cheminformatics with RDKit in Python: HSC_DSW - October, 2021**

The contents of this notebook were drawn from inspiration by the excellent talk and blog post by Pat Walters$^1$ and the Data Professor$^2$.

The machine learning example described is also briefly described in the book ***Deep Learning for the Life Sciences: Applying Deep Learning to Genomics, Microscopy, Drug Discovery, and More***.$^3$

In this Jupyter notebook, we will be going through some functions of RDKit for Cheminformatics (What's possible with RDKit). We will also build a simple decision tree to predict solubility of molecules in the original Delaney dataset$^4$.

In [1]:
#@title **Install Miniconda, RDKit and other libraries**
#@markdown Please execute this cell by pressing the _Play_ button
#@markdown on the left. Installation may take a few minutes.
#@markdown Double click on this text to show/hide the installation script

from IPython.utils import io
import tqdm.notebook

total = 50

with tqdm.notebook.tqdm(total=total) as pbar:
    with io.capture_output() as captured:

      # install anaconda
      %shell wget https://repo.anaconda.com/miniconda/Miniconda3-py37_4.8.2-Linux-x86_64.sh
      pbar.update(10)
      %shell chmod +x Miniconda3-py37_4.8.2-Linux-x86_64.sh
      %shell bash ./Miniconda3-py37_4.8.2-Linux-x86_64.sh -b -f -p /usr/local
      %shell conda install -c rdkit rdkit -y
      pbar.update(30)

      import sys
      sys.path.append('/usr/local/lib/python3.7/site-packages/')

      # non rdkit installations
      %shell pip install mols2grid
      %shell pip install dtreeviz
      %shell pip install git+https://github.com/PatWalters/clusterama.git #To perform clustering



      pbar.update(10)

  0%|          | 0/50 [00:00<?, ?it/s]

In [12]:
# install conda
!pip install -q condacolab
import condacolab
condacolab.install()

⏬ Downloading https://github.com/jaimergp/miniforge/releases/latest/download/Mambaforge-colab-Linux-x86_64.sh...


HTTPError: HTTP Error 404: Not Found

In [5]:
!conda install rdkit

Solving environment: / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | done


==> WARNING: A newer version of conda exists. <==
  current version: 23.1.0
  latest version: 25.1.1

Please update conda by running

    $ conda update -n base -c defaults conda

Or to minimize the number of packages updated during conda update use

     conda install conda=25.1.1



# All requested packages already installed.



In [10]:
import rdkit

ImportError: /usr/local/lib/python3.7/site-packages/rdkit/../../../libboost_python37.so.1.67.0: undefined symbol: _Py_fopen

In [7]:
#@title **Import the necessary Python libraries**
#@markdown Execute this cell by clicking the _Play_ button the left

## general and data handling
import numpy as np
import pandas as pd
import os
from collections import Counter


# Required RDKit and other modules
from rdkit import Chem #RDKit Chemistry
from rdkit.Chem.Draw import IPythonConsole #RDKit drawing
from rdkit.Chem import Draw #RDKit drawing
# A few settings to improve the quality of structures
from rdkit.Chem import rdDepictor
IPythonConsole.ipython_useSVG = True
rdDepictor.SetPreferCoordGen(True)
from rdkit.Chem import PandasTools #Add the ability to add a molecule to a dataframegrid
import mols2grid #The mols2grid library provides a convenient way of displaying molecules in a grid

from rdkit.Chem import Draw, rdFMCS, AllChem
from clusterama import ButinaCluster, display_cluster_members


# modeling
import pandas as pd #data table manipulation
from rdkit import Chem # basic cheminformatics
from rdkit.Chem.Descriptors import MolWt, MolLogP, NumAromaticRings, NumHDonors, NumHAcceptors
import math #needed for log10
import seaborn as sns #plotting
from sklearn.tree import DecisionTreeClassifier, plot_tree # descision trees
from sklearn.model_selection import train_test_split # split a dataset
from tqdm import tqdm # progress bar
from matplotlib import pyplot as plt # plotting
from dtreeviz.trees import * #plotting decision trees
from sklearn.metrics import roc_auc_score, plot_roc_curve, plot_confusion_matrix # model stats

tqdm.pandas()


# Graphing
import matplotlib.pyplot as plt
import seaborn as sns

ImportError: /usr/local/lib/python3.7/site-packages/rdkit/../../../libboost_python37.so.1.67.0: undefined symbol: _Py_fopen

## 1. **Simple RDKit functions**

Create a molecule (benzene) from a SMILES string

In [ ]:
mol = Chem.MolFromSmiles("c1ccccc1")
mol

Get SMILES of Imatinib (Gleevec)
*   From [ChEMBL](https://www.ebi.ac.uk/chembl/compound_report_card/CHEMBL941/)
*   From [Wikipedia](https://en.wikipedia.org/wiki/Imatinib)

In [ ]:
glvc = Chem.MolFromSmiles("CN1CCN(Cc2ccc(cc2)C(=O)Nc3ccc(C)c(Nc4nccc(n4)c5cccnc5)c3)CC1")
glvc

Get SMILES of Tylenol
*   From [Wikipedia](https://en.wikipedia.org/wiki/Paracetamol)

In [ ]:
tylenol = Chem.MolFromSmiles("??")
tylenol

Read a set of molecules from an SD file

In [ ]:
!wget https://raw.githubusercontent.com/francisacquah466/DSW2021-Intro-to-Cheminformatics-ML/main/example_compounds.sdf

Check the first 10 lines of the sd file

In [ ]:
!head example_compounds.sdf

In [ ]:
mols = [x for x in Chem.SDMolSupplier("example_compounds.sdf")]

In [ ]:
mols

Draw these molecules as a grid

In [ ]:
Draw.MolsToGridImage(mols,molsPerRow=6,useSVG=True)

We can use the mols2grid library to display molecules in a grid

In [ ]:
mols2grid.display(mols)

In [ ]:
mols2grid.get_selection()

We can also read an SD file into a Pandas dataframe.

In [ ]:
df = PandasTools.LoadSDF("example_compounds.sdf")

In [ ]:
df.head()

Let's add columns with molecular weight and LogP to the dataframe.

In [ ]:
from rdkit.Chem.Descriptors import MolWt
from rdkit.Chem.Crippen import MolLogP
df['MW'] = [MolWt(x) for x in df.ROMol]
df['LogP'] = [MolLogP(x) for x in df.ROMol]

In [ ]:
df.head()

We can use a boxplot to examine the distribution of molecular weight within the dataframe.

In [ ]:
ax = sns.boxplot(x=df.MW)

## 2. **Building a Simple Decision Tree**

A simple function to calculate molecular weight, LogP, number of aromatic rings, number of hydrogen bond donors and acceptors from a SMILES

In [ ]:
def calc_descriptors(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol:
        mw, logp, num_arom_rings, hbd, hba = [x(mol) for x in [MolWt, MolLogP, NumAromaticRings, NumHDonors, NumHAcceptors]]
        res = [mw, logp, num_arom_rings, hbd, hba]
    else:
        res = [None] * 5
    return res

# Read Delaney's solubility dataset

The full paper can be found: [ESOL:  Estimating Aqueous Solubility Directly from Molecular Structure](https://pubs.acs.org/doi/10.1021/ci034243x)

In [ ]:
!wget https://raw.githubusercontent.com/francisacquah466/DSW2021-Intro-to-Cheminformatics-ML/main/delaney.csv

In [ ]:
df = pd.read_csv("delaney.csv")

Get the dataframe column names

In [ ]:
df.head()

In [ ]:
df.columns

Change the name of column 1 to "LogS"

In [ ]:
cols = list(df.columns)
cols[1] = 'LogS'
df.columns = cols

Add a new column "IsSol" to indicate whether a molecule's solubility is greater than 200uM


In [ ]:
df['IsSol'] = df.LogS > math.log10(200 * 1e-6)

In [ ]:
df.head()

In [ ]:
sns.displot(x='LogS',hue="IsSol",data=df)

Add the descriptors to the dataframe. Note that all of the descriptors are going into one column called "desc".

In [ ]:
df['desc'] = df.SMILES.progress_apply(calc_descriptors)

In [ ]:
df.head()

Split the descriptors into their on own columns.

In [ ]:
desc_cols = ['MW','LogP','NumAromatic','HBD','HBA']
df[desc_cols] = df.desc.to_list()

In [ ]:
df.head()

Split the data into training and test sets.

In [ ]:
train, test = train_test_split(df)

In [ ]:
train.shape, test.shape

Look again at the X variables

In [ ]:
desc_cols

Split the training and test sets into X and y variables.

In [ ]:
train_X = train[desc_cols]
train_y = train.IsSol
test_X = test[desc_cols]
test_y = test.IsSol


Create and train a classifier

In [ ]:
cls = DecisionTreeClassifier(max_depth=2)
cls.fit(train_X,train_y)


Predict on the test set

In [ ]:
pred = cls.predict(test_X)
auc = roc_auc_score(test_y, pred)
print(f"ROC AUC = {auc:.2f}")


Plot a confusion matrix to show the classifier performance

In [ ]:
plot_confusion_matrix(cls,test_X,test_y)

Plot an ROC cure to show the classifier performance

In [ ]:
plot_roc_curve(cls,test_X,test_y)

Use the default view from SciKit Learn to plot the decision tree

Use dtreeviz to plot the decision tree

In [ ]:
viz = dtreeviz(cls, train_X, train_y, feature_names = desc_cols,
               target_name = "Solubility",class_names=["False","True"],scale=2)
viz

Can the classifier be used to classify new molecules as soluble or not??

In [ ]:
tylenol_sol = calc_descriptors('CC(=O)Nc1ccc(O)cc1')
tylenol_sol

In [ ]:
print(cls.predict((tylenol_sol,)))
print(cls.predict_proba((tylenol_sol,)))

In [ ]:
glvc_sol = calc_descriptors("CN1CCN(Cc2ccc(cc2)C(=O)Nc3ccc(C)c(Nc4nccc(n4)c5cccnc5)c3)CC1")
glvc_sol

In [ ]:
print(cls.predict((glvc_sol,)))
print(cls.predict_proba((glvc_sol,)))

## 3. **Clustering a library of molecules**

Clustering 1

In [ ]:
butina_cluster = ButinaCluster("rdkit")
df = pd.read_csv("delaney.csv")
df['Cluster'] = butina_cluster.cluster_smiles(df.SMILES,sim_cutoff=0.7)
cluster_rows = []
for k,v in df.groupby("Cluster"):
    cluster_rows.append([v.SMILES.values[0],k,len(v)])
cluster_df = pd.DataFrame(cluster_rows,columns=["SMILES","Cluster","Num"])
df.head()

In [ ]:
cols = list(df.columns)
cols[0] = 'Name'
df.columns = cols

In [ ]:
df.head()

In [ ]:
len(df)

In [ ]:
cluster_df

In [ ]:
mols2grid.display(cluster_df,subset=["img","Num"])

In [ ]:
display_cluster_members(df,mols2grid.get_selection().keys(),True)

# **References**

1. [Pat Walters Github](https://github.com/PatWalters/chem_tutorial)
2. [Data Professor YouTube](https://www.youtube.com/c/DataProfessor)
3. [Deep Learning for the Life Sciences](https://www.oreilly.com/library/view/deep-learning-for/9781492039822/)
4.[ESOL:  Estimating Aqueous Solubility Directly from Molecular Structure](https://pubs.acs.org/doi/10.1021/ci034243x)